# Blindspot · GRPO Failure Analysis

This notebook is the **public repo-backed version** of the GRPO failure analysis for Blindspot.

It does two things:
1. Summarizes the observed training dynamics from the failed GRPO run.
2. Shows how that run maps onto the reproducible training script in `training/grpo_train.py`.

The key point is simple: GRPO did not just underperform here. It collapsed into a low-variance, low-entropy policy with almost no learning signal.

## 1. What the failed run looked like

From the 150-step GRPO run (600 total rollouts):

- First 10% mean reward: **-0.004**
- Last 10% mean reward: **0.000**
- Net gain: **+0.004**
- Many steps had **reward_std = 0.000**

That is the signature of within-group advantage collapse: the rollouts stop being meaningfully rankable, so policy updates have almost nothing to work with.

In [ ]:
import pandas as pd

log_rows = [
    {"step": 5, "reward": 0.0000, "reward_std": 0.0000},
    {"step": 45, "reward": 0.0000, "reward_std": 0.0000},
    {"step": 50, "reward": 0.0000, "reward_std": 0.0000},
    {"step": 55, "reward": 0.0000, "reward_std": 0.0000},
    {"step": 65, "reward": 0.0000, "reward_std": 0.0000},
    {"step": 75, "reward": 0.0000, "reward_std": 0.0000},
    {"step": 85, "reward": 0.0000, "reward_std": 0.0000},
    {"step": 90, "reward": 0.0000, "reward_std": 0.0000},
    {"step": 105, "reward": 0.0000, "reward_std": 0.0000},
    {"step": 110, "reward": 0.0000, "reward_std": 0.0000},
    {"step": 125, "reward": 0.0000, "reward_std": 0.0000},
    {"step": 130, "reward": 0.0000, "reward_std": 0.0000},
    {"step": 140, "reward": 0.0000, "reward_std": 0.0000},
    {"step": 145, "reward": 0.0000, "reward_std": 0.0000},
    {"step": 150, "reward": 0.0000, "reward_std": 0.0000},
]

df = pd.DataFrame(log_rows)
df

## 2. The policy output that exposed the collapse

The most telling qualitative symptom was that the model kept producing the same action sequence:

```json
{"type": "surface", "concept_id": 1}
{"type": "surface", "concept_id": 1}
{"type": "surface", "concept_id": 1}
... (repeated, then stop)
```

Once the rollouts became this homogeneous, GRPO had almost no relative signal left inside each group.

In [ ]:
sample_actions = [
    {"type": "surface", "concept_id": 1},
    {"type": "surface", "concept_id": 1},
    {"type": "surface", "concept_id": 1},
    {"type": "surface", "concept_id": 1},
]
sample_actions

## 3. Why Blindspot is hard for GRPO from scratch

Blindspot stacks three conditions that make early collapse likely:

- **Delayed reward**: useful signal only shows up at the end of the episode.
- **False-positive penalty**: the model is punished for recommending the wrong concept.
- **Strong personalization**: the correct action depends heavily on the user profile.

That combination makes low-entropy behavior locally attractive for a weak base policy.

## 4. Public reproduction path

The GRPO training script used for this setup lives in the repo at `training/grpo_train.py`.

If you want to rerun the setup locally or in Colab, the high-level flow is:

1. Start the Blindspot OpenEnv server.
2. Point `ENV_URL` at that running server.
3. Run `python training/grpo_train.py --base-model unsloth/Qwen3.5-9B`.

The important result is not that GRPO got a small reward. It is that the policy collapsed before exploration ever got going.

In [ ]:
from pathlib import Path

script_path = Path('/content/blindspot-env/training/grpo_train.py')
print('Expected repo script:', script_path)
print('Example command:')
print('python training/grpo_train.py --base-model unsloth/Qwen3.5-9B')

## 5. Takeaway

This is why the project uses SFT as the warm start. Once the policy begins producing diverse action sequences, RL has something to rank. Before that, GRPO is mostly staring at copies of the same bad rollout.